# Section 1: Initial Dataset Inspection and Loading
In this step, we load the raw `ExtractedTweets.csv` dataset, inspect the total record count, view the column schemas, check for null/missing values, and evaluate the initial distribution of party labels.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Construct path to the target CSV file
# Using the known kagglehub download location directly
path = '/root/.cache/kagglehub/datasets/kapastor/democratvsrepublicantweets/versions/4'
target_file = Path(path) / 'ExtractedTweets.csv'

# Load dataset
df_raw = pd.read_csv(target_file)
print(f"Initial Dataset Shape: {df_raw.shape}")
print(\"\
--- Dataset Columns and Types ---\")
df_raw.info()

print(\"\
--- Missing Values Count per Column ---\")
print(df_raw.isnull().sum())

print(\"\
--- Unique Party Labels ---\")
print(df_raw['Party'].value_counts(dropna=False))

# Section 2: Data Cleaning and Label Normalization
Here, we sanitize our dataset by:
1. Removing empty or null tweets and labels.
2. Verifying and normalizing party labels (retaining only `Democrat` and `Republican`).
3. Resolving cases where duplicate tweets have conflicting labels (e.g., same tweet text appearing under different party designations) to prevent class noise and data leakage.

In [ ]:
# Initial setup for tracking row elimination
initial_count = len(df_raw)
reports = []

# 1. Drop records with missing values in Tweet or Party
df_clean = df_raw.dropna(subset=['Tweet', 'Party']).copy()
drop_nulls = initial_count - len(df_clean)
reports.append(f"Dropped {drop_nulls} row(s) due to missing 'Tweet' or 'Party' values.")

# 2. Normalize and check Party labels
valid_labels = {'Democrat', 'Republican'}
unexpected_labels = set(df_clean['Party'].unique()) - valid_labels
if unexpected_labels:
    print(f"Warning: Found unexpected party labels: {unexpected_labels}")
    pre_filter = len(df_clean)
    df_clean = df_clean[df_clean['Party'].isin(valid_labels)]
    reports.append(f"Dropped {pre_filter - len(df_clean)} row(s) with unexpected labels: {unexpected_labels}")
else:
    reports.append("All party labels are successfully normalized to 'Democrat' or 'Republican'. No rows dropped here.")

# 3. Handle identical tweets with conflicting party labels
# Group by tweet text to identify conflicting labels
tweet_groups = df_clean.groupby('Tweet')['Party'].nunique()
conflicting_tweets = tweet_groups[tweet_groups > 1].index

if len(conflicting_tweets) > 0:
    print(f"Found {len(conflicting_tweets)} unique tweets with conflicting labels.")
    pre_conflict_count = len(df_clean)
    df_clean = df_clean[~df_clean['Tweet'].isin(conflicting_tweets)]
    dropped_conflicts = pre_conflict_count - len(df_clean)
    reports.append(f"Dropped {dropped_conflicts} row(s) due to conflicting party labels for identical tweet texts.")
else:
    reports.append("No tweets with conflicting labels were detected.")

# 4. Handle exact duplicates (identical tweet, handle, and label)
pre_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Tweet', 'Party', 'Handle'])
reports.append(f"Dropped {pre_dedup - len(df_clean)} exact duplicate rows.")

# Print cleaning report summary
print(\"\
=== Data Cleaning Summary ===\")
for rep in reports:
    print("- ", rep)
print(f"Final Cleaned Dataset Size: {df_clean.shape}")

# Section 3: Text Preprocessing Variants
To satisfy requirement 5, we keep the original text untouched and implement three distinct preprocessing variants without applying stopword removal, stemming, or lemmatization.

### Explanation of Potential Preprocessing Effects:
* **Stop words**: Retaining them keeps syntactical structures and style characteristics that are often crucial indicators of political writing style or syntactic fingerprints.
* **Stemming/Lemmatization**: Skipping this retains the precise grammatical tense and conjugation of verbs, which may correlate with speech patterns of different parties, avoiding loss of nuance in political messaging.
* **Variant A**: Normalizes formatting style (lowercase, spacing) while keeping punctuation intact. Punctuations like `!` or `?` might reveal high emotion or rhetorical styles.
* **Variant B**: Cleans general punctuation but retains alphanumeric content associated with hashtags (`#`) and mentions (`@`), preserving categorical community and topic targets.
* **Variant C**: Completely isolates standard natural language by dropping URLs, mentions, hashtag text, and all punctuation.

In [ ]:
import re
import string

# Preserve untouched copy
df_clean['tweet_original'] = df_clean['Tweet'].copy()

# Variant A: lowercase text and normalize whitespace only
def preprocess_variant_a(text):
    text = str(text).lower()
    # Normalize whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Variant B: remove punctuation while retaining the words that follow hashtags (#) and mentions (@)
def preprocess_variant_b(text):
    text = str(text).lower()
    # Strip out URLs first to prevent URL punctuation from leaving remnants
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    # Keep words following hashtags and mentions but remove the prefix symbol
    text = re.sub(r'[@#](\w+)', r'\1', text)
    # Translate punctuation to whitespace, then clean extra spaces
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Variant C: remove URLs, user mentions, hashtag markers/hashtags, and punctuation
def preprocess_variant_c(text):
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    # Remove completely the mention (@username) and the hashtag words (#topic)
    text = re.sub(r'[@#]\w+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Normalize whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the text variants
print("Applying preprocessing variants...")
df_clean['tweet_variant_a'] = df_clean['tweet_original'].apply(preprocess_variant_a)
df_clean['tweet_variant_b'] = df_clean['tweet_original'].apply(preprocess_variant_b)
df_clean['tweet_variant_c'] = df_clean['tweet_original'].apply(preprocess_variant_c)

print("Preprocessing completed. Displaying small preview sample of variants:")
display(df_clean[['tweet_original', 'tweet_variant_a', 'tweet_variant_b', 'tweet_variant_c']].head(3))

# Section 4: Data Leakage Prevention and Stratified Splitting
We must prevent identical tweet text from crossing train, validation, and test splits (which causes data leakage and overoptimistic performance). To address this, we drop duplicate text records in our preprocessing splits and apply a stratified partition to preserve the original class ratios.

In [ ]:
from sklearn.model_selection import train_test_split

# Drop duplicates based on the original tweet text to guarantee no cross-split leakage
df_unique = df_clean.drop_duplicates(subset=['tweet_original']).copy()
print(f"Dataset size after dropping duplicate tweet text instances: {df_unique.shape[0]} rows (down from {df_clean.shape[0]})")

# We will do a 80-10-10 split (Train: 80%, Val: 10%, Test: 10%)
# First, split into 80% Train and 20% Temp
train_df, temp_df = train_test_split(
    df_unique,
    test_size=0.2,
    random_state=42,
    stratify=df_unique['Party']
)

# Split Temp into 50% Val and 50% Test (each representing 10% of total)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['Party']
)

print(f"\nTrain partition shape: {train_df.shape} ({len(train_df)/len(df_unique)*100:.1f}%)")
print(f"Val partition shape: {val_df.shape} ({len(val_df)/len(df_unique)*100:.1f}%)")
print(f"Test partition shape: {test_df.shape} ({len(test_df)/len(df_unique)*100:.1f}%)")

print(\"\
--- Stratified Label Distributions ---\")
print("Train: ", train_df['Party'].value_counts(normalize=True).to_dict())
print("Val:   ", val_df['Party'].value_counts(normalize=True).to_dict())
print("Test:  ", test_df['Party'].value_counts(normalize=True).to_dict())

# Section 5: Saving Processed Datasets to `/content/`
We save our cleaned overall dataset and individual stratified splits as local CSV files inside `/content/` so they are immediately available for downstream model training tasks.

In [ ]:
output_dir = Path('/content')
output_dir.mkdir(parents=True, exist_ok=True)

# Save files
df_clean.to_csv(output_dir / 'cleaned_dataset_all_variants.csv', index=False)
train_df.to_csv(output_dir / 'train_split.csv', index=False)
val_df.to_csv(output_dir / 'val_split.csv', index=False)
test_df.to_csv(output_dir / 'test_split.csv', index=False)

print("Files successfully saved:")
print(f"- All variants full set: {output_dir / 'cleaned_dataset_all_variants.csv'}")
print(f"- Train split: {output_dir / 'train_split.csv'}")
print(f"- Validation split: {output_dir / 'val_split.csv'}")
print(f"- Test split: {output_dir / 'test_split.csv'}")